# Wagner Questionnaire Demo

This notebook asks Wagner questions interactively in the notebook and scores the response.

Run cells in order:
1. Cell 2: Load package (optional sanity check)
2. Cell 3: Answer questions (`y/yes` or `n/no`) and view scored output

In [ ]:
const path = require('path');

function loadWagnerModule() {
  const candidates = [
    '.',
    './src/index.js',
    './collection/wagner',
    './collection/wagner/src/index.js'
  ];

  for (const candidate of candidates) {
    try {
      return require(path.resolve(candidate));
    } catch (error) {
      // Try next candidate path.
    }
  }

  throw new Error('Could not load Wagner module from expected paths.');
}

const { runQuestionnaire, analyzeQuestionnaireResponse, QUESTIONS } = loadWagnerModule();

console.log(`Loaded Wagner module with ${QUESTIONS.length} questions.`);

In [ ]:
const readline = require('node:readline');
const { stdin, stdout } = require('node:process');

async function promptLine(promptText) {
  if (globalThis.$$ && typeof globalThis.$$.input === 'function') {
    return globalThis.$$.input({ prompt: promptText });
  }

  return new Promise((resolve) => {
    const rl = readline.createInterface({
      input: stdin,
      output: stdout
    });

    rl.question(promptText, (answer) => {
      rl.close();
      resolve(answer);
    });
  });
}

async function askYesNo(question) {
  console.log(`\n${question.id}: ${question.text}`);

  const definitionEntries = Object.entries(question.definitions || {});
  if (definitionEntries.length > 0) {
    console.log('Definitions:');
    for (const [term, info] of definitionEntries) {
      console.log(`- ${term}: ${info.definition}`);
    }
  }

  while (true) {
    const raw = await promptLine('Your answer [y/n]: ');
    const value = String(raw || '').trim().toLowerCase();
    if (value === 'y' || value === 'yes') {
      return true;
    }
    if (value === 'n' || value === 'no') {
      return false;
    }
    console.log("Please answer with 'y'/'yes' or 'n'/'no'.");
  }
}

(async () => {
  console.log('Starting interactive Wagner questionnaire...');
  globalThis.questionnaireOutcome = await runQuestionnaire(askYesNo);
  globalThis.questionnaireAnalysis = analyzeQuestionnaireResponse(globalThis.questionnaireOutcome);

  console.log('\nQuestionnaire outcome:');
  console.log(JSON.stringify(globalThis.questionnaireOutcome, null, 2));

  console.log('\nScored analysis:');
  console.log(JSON.stringify(globalThis.questionnaireAnalysis, null, 2));
})();

## Another Way in Binder: Run the CLI in a Terminal or Code Cell

In Binder, you can run the same interactive CLI directly from a Terminal tab:

1. Open a Terminal in Binder (Launcher -> Terminal).
2. Run:

```bash
cd collection/wagner
npx wagner
```

You can also run the next JavaScript cell, which is the JS-kernel equivalent of a shell bang command.

In [ ]:
const { execSync } = require('node:child_process');

execSync('cd collection/wagner && npx wagner', {
  stdio: 'inherit',
  shell: true
});